In [1]:
import numpy as np
import math
import time
import warnings
warnings.filterwarnings('ignore')

In [2]:
def f(x, y):
    return x**2 * y**2 * math.log(8 * x**2 + 3 * y**2)

def gradient(x, y):
    df_dx = 2 * x * y**2 * math.log(8 * x**2 + 3 * y**2) + (16 * x**3 * y**2) / (8 * x**2 + 3 * y**2)
    df_dy = 2 * x**2 * y * math.log(8 * x**2 + 3 * y**2) + (6 * x**2 * y**3) / (8 * x**2 + 3 * y**2)
    return df_dx, df_dy

# Стационарная точка А - локальный минимум функции
x_a = 1 / (4 * math.e ** 0.25)
y_a = 1 / (math.sqrt(6) * math.e ** 0.25)
extremum_point_A = (x_a, y_a)
# Стационарная точка C - локальный минимум функции
x_c = -x_a
y_c = y_a
extremum_point_C = (x_c, y_c)

last_points = []

def export_trajectory(filename):
    """ Экспорт траектории градиентного спуска в TSV файл для визуализации """
    with open(filename, 'w') as fh:
        fh.write('x\ty\tz\n')
        for x, y in last_points:
            fh.write(f'{x}\t{y}\t{f(x, y)}\n')

def gradient_descent(start_point, a_0, num_iterations,
                     normalize=False, step_mode='const',
                     stop_criterion = 'none', q=0.99, lin_rate=0.01,
                     epsilon=1e-6, delta=1e-6):

    """Градиентный спуск с различными режимами шага и критериями остановки"""

    global last_points
    t0 = time.perf_counter()
    list_of_points = [start_point]
    a_k = a_0
    f_prev = f(*start_point)
    converged = False
    delta_f_final = None
    delta_x_final = None

    for i in range(num_iterations):
        if step_mode == 'geom':
            a_k = a_0 * q ** i
        elif step_mode == 'linear':
            a_k = a_0 / (1 + i * lin_rate)
        grad = np.array(gradient(*list_of_points[-1]))

        if normalize:
            grad = grad / np.linalg.norm(grad)
        x_k, y_k = list_of_points[-1] - a_k * grad # Вычисляем новую точку
        list_of_points.append((x_k, y_k))

        delta_x_final = np.linalg.norm(np.array(list_of_points[-1]) - np.array(list_of_points[-2])) # Вычисляем изменение в координатах
        f_curr = f(*list_of_points[-1])
        delta_f_final = abs(f_curr - f_prev) # Вычисляем изменение в значении функции

        if stop_criterion == 'delta_f':
            if delta_f_final < epsilon:
                converged = True
                break

        elif stop_criterion == 'delta_x':
            if delta_x_final < delta:
                converged = True
                break
        f_prev = f_curr
    elapsed = time.perf_counter() - t0

    last_points = list_of_points
    err_point = np.linalg.norm(np.array(list_of_points[-1]) - np.array(extremum_point_A))
    err_value = abs(f(*list_of_points[-1]) - f(*extremum_point_A))
    reason = 'Сходимость' if converged else 'Макс. итераций'
    final_point = list_of_points[-1]
    return (
        f"Старт:           {start_point}\n"
        f"Шаг:             {step_mode}\n"
        f"Критерий:        {stop_criterion}\n"
        f"Итераций:        {i + 1}\n"
        f"Время:           {elapsed*1000:.2f} мс\n"
        f"Результат:       {reason}\n"
        f"Найденная точка: ({float(final_point[0]):.6f}, {float(final_point[1]):.6f})\n"
        f"Точная точка:    ({extremum_point_A[0]:.6f}, {extremum_point_A[1]:.6f})\n"
        f"Ошибка по точке: {float(err_point):.3e}\n"
        f"f(найденная):    {float(f(*final_point)):.6e}\n"
        f"f(точная):       {f(*extremum_point_A):.6e}\n"
        f"Ошибка по f:     {float(err_value):.3e}\n"
        f"|delta f|:       {float(delta_f_final):.3e}\n"
        f"|delta x|:       {float(delta_x_final):.3e}"
    )

In [3]:
print("Проверка градиента в точке А: ", gradient(x_a, y_a))
print("Координаты точки А: ", x_a, y_a, f(x_a, y_a))
print("Координаты точки C: ", x_c, y_c, f(x_c, y_c))

Проверка градиента в точке А:  (3.469446951953614e-18, 8.673617379884035e-18)
Координаты точки А:  0.19470019576785122 0.31794408830081894 -0.0019160387561012622
Координаты точки C:  -0.19470019576785122 0.31794408830081894 -0.0019160387561012622


## Эксперименты

Ниже приведена серия экспериментов с различными комбинациями параметров с целью продемонстрировать поведение метода в разных режимах и обосновать выбор параметров для главного эксперимента.

В каждом эксперименте выводятся: параметры запуска, число итераций, время работы, найденная точка, отклонение от эталона $A$ и значения приращений на момент остановки.

In [4]:
# Эксперимент 1
# Градиент с постоянным шагом, критерий delta_f
print(gradient_descent((10, 10), 1, 100, stop_criterion="delta_f"))
# Постоянный шаг слишком большой и точки расходятся в бесконечность

Старт:           (10, 10)
Шаг:             const
Критерий:        delta_f
Итераций:        100
Время:           0.56 мс
Результат:       Макс. итераций
Найденная точка: (nan, nan)
Точная точка:    (0.194700, 0.317944)
Ошибка по точке: nan
f(найденная):    nan
f(точная):       -1.916039e-03
Ошибка по f:     nan
|delta f|:       nan
|delta x|:       nan


In [5]:
# Эксперимент 2
# Нормированный градиент, линейное убывание шага, критерий delta_x
print(gradient_descent((10, 10), 1, 1000, normalize=True, step_mode='linear', stop_criterion='delta_x'))
# Критерий остановки не сработал, точки не сошлись к А, отклонение большое

Старт:           (10, 10)
Шаг:             linear
Критерий:        delta_x
Итераций:        1000
Время:           7.40 мс
Результат:       Макс. итераций
Найденная точка: (0.203728, -0.315581)
Точная точка:    (0.194700, 0.317944)
Ошибка по точке: 6.336e-01
f(найденная):    -1.904504e-03
f(точная):       -1.916039e-03
Ошибка по f:     1.153e-05
|delta f|:       7.539e-04
|delta x|:       9.099e-02


In [6]:
# Эксперимент 3
# Нормированный градиент, маленький шаг, критерий delta_f
print(gradient_descent((10, 10), 0.1, 1000, normalize=True, stop_criterion="delta_f"))
# Y точек сходится к точке C, но X точек осциллирует вокруг точки C

Старт:           (10, 10)
Шаг:             const
Критерий:        delta_f
Итераций:        1000
Время:           8.20 мс
Результат:       Макс. итераций
Найденная точка: (-0.244104, 0.319007)
Точная точка:    (0.194700, 0.317944)
Ошибка по точке: 4.388e-01
f(найденная):    -1.491191e-03
f(точная):       -1.916039e-03
Ошибка по f:     4.248e-04
|delta f|:       7.851e-05
|delta x|:       1.000e-01


In [7]:
# Эксперимент 4
# Нормированный градиент, геометрическое убывание шага, epsilon = delta = 0.001, критерий delta_x
print(gradient_descent((10, 10), 1, 1000, normalize=True, step_mode='geom', stop_criterion='delta_x', epsilon=1e-3, delta=1e-3))
# Точки практически сошлись к точке А, критерий delta_x остановил спуск

Старт:           (10, 10)
Шаг:             geom
Критерий:        delta_x
Итераций:        689
Время:           5.89 мс
Результат:       Сходимость
Найденная точка: (0.195020, 0.318038)
Точная точка:    (0.194700, 0.317944)
Ошибка по точке: 3.329e-04
f(найденная):    -1.916021e-03
f(точная):       -1.916039e-03
Ошибка по f:     1.785e-08
|delta f|:       5.209e-08
|delta x|:       9.931e-04


In [8]:
# Эксперимент 5
# Нормированный градиент, убывающий шаг (геометрическая прогрессия), оба критерия остановки, epsilon = delta = 1e-6
print(gradient_descent((10, 10), 1, 1000, normalize=True, step_mode='geom', stop_criterion='delta_x'))
print()
print(gradient_descent((10, 10), 1, 1000, normalize=True, step_mode='geom', stop_criterion='delta_f'))
# Критерий delta_x не сработал, но точки сошлись к точке А, критерий delta_f сработал, но отклонение от точки А очень большое

Старт:           (10, 10)
Шаг:             geom
Критерий:        delta_x
Итераций:        1000
Время:           7.84 мс
Результат:       Макс. итераций
Найденная точка: (0.194741, 0.317956)
Точная точка:    (0.194700, 0.317944)
Ошибка по точке: 4.270e-05
f(найденная):    -1.916038e-03
f(точная):       -1.916039e-03
Ошибка по f:     2.933e-10
|delta f|:       2.931e-10
|delta x|:       4.361e-05

Старт:           (10, 10)
Шаг:             geom
Критерий:        delta_f
Итераций:        116
Время:           0.75 мс
Результат:       Сходимость
Найденная точка: (0.026559, 0.113500)
Точная точка:    (0.194700, 0.317944)
Ошибка по точке: 2.647e-01
f(найденная):    -2.832390e-05
f(точная):       -1.916039e-03
Ошибка по f:     1.888e-03
|delta f|:       8.166e-08
|delta x|:       3.148e-01


In [9]:
# Эксперимент 6
# Нормированный градиент, убывающий шаг (геометрическая прогрессия), точка (7, 7)
print(gradient_descent((7, 7), 1, 1000, normalize=True, step_mode='geom', stop_criterion='delta_x'))
# Точки сходятся к точке C, но критерии остановки не сработали.

Старт:           (7, 7)
Шаг:             geom
Критерий:        delta_x
Итераций:        1000
Время:           8.27 мс
Результат:       Макс. итераций
Найденная точка: (-0.194687, 0.317940)
Точная точка:    (0.194700, 0.317944)
Ошибка по точке: 3.894e-01
f(найденная):    -1.916039e-03
f(точная):       -1.916039e-03
Ошибка по f:     2.955e-11
|delta f|:       1.157e-10
|delta x|:       4.361e-05


In [10]:
# Эксперимент 7
# Нормированный градиент, убывающий шаг (геометрическая прогрессия), точка (10, 5)
print(gradient_descent((10, 5), 1, 1000, normalize=True, step_mode='geom', stop_criterion='delta_x'))
# Точки сошлись к (3.81, 0) - оси X, нестрогому локальному минимуму вне эллипса 8x^2 + 3y^2 = 1

Старт:           (10, 5)
Шаг:             geom
Критерий:        delta_x
Итераций:        1000
Время:           12.85 мс
Результат:       Макс. итераций
Найденная точка: (3.819640, 0.000043)
Точная точка:    (0.194700, 0.317944)
Ошибка по точке: 3.639e+00
f(найденная):    1.293722e-07
f(точная):       -1.916039e-03
Ошибка по f:     1.916e-03
|delta f|:       1.294e-07
|delta x|:       4.361e-05


In [11]:
# Эксперимент 8
# Нормированный градиент, убывающий шаг (геометрическая прогрессия), критерий delta_x, epsilon = delta = 1e-4
print(gradient_descent((10,10), 1, 1000, normalize=True, step_mode='geom', stop_criterion='delta_x', epsilon = 1e-4, delta=1e-4))
# Точки сошлись к точке А, критерий delta_x сработал при delta = 1e-4, когда при delta = 1e-6 не сработал, но точки все равно сошлись к А

Старт:           (10, 10)
Шаг:             geom
Критерий:        delta_x
Итераций:        918
Время:           7.60 мс
Результат:       Сходимость
Найденная точка: (0.194768, 0.317964)
Точная точка:    (0.194700, 0.317944)
Ошибка по точке: 7.046e-05
f(найденная):    -1.916038e-03
f(точная):       -1.916039e-03
Ошибка по f:     7.988e-10
|delta f|:       6.639e-10
|delta x|:       9.942e-05


## Сводка экспериментов

| Эксперимент | Шаг | Критерий | Итер. | Результат                   |              Ошибка по точке |
|---|---|---|---:|-----------------------------|-----------------------------:|
| 1. const, $a_0=1$, без нормировки | const | $\|\Delta f\|<10^{-6}$ | 100 | расходится                  |                            — |
| 2. linear, $a_0=1$, нормировка | linear | $\|\Delta x\|<10^{-6}$ | 1000 | не сошлось, макс. итераций  |          $6.3 \cdot 10^{-1}$ |
| 3. const, $a_0=0.1$, нормировка | const | $\|\Delta f\|<10^{-6}$ | 1000 | осцилляция                  |          $4.4 \cdot 10^{-1}$ |
| 4. geom, $a_0=1$, $\varepsilon=\delta=10^{-3}$ | geom | $\|\Delta x\|<10^{-3}$ | 689 | сошлось ($A$)               |          $3.3 \cdot 10^{-4}$ |
| 5а. geom, $a_0=1$, $\delta=10^{-6}$ | geom | $\|\Delta x\|<10^{-6}$ | 1000 | максимум итераций ($A$)     |          $4.3 \cdot 10^{-5}$ |
| 5б. geom, $a_0=1$, $\varepsilon=10^{-6}$ | geom | $\|\Delta f\|<10^{-6}$ | 116 | остановилось преждевременно |          $2.6 \cdot 10^{-1}$ |
| 6. geom, старт $(7,7)$ | geom | $\|\Delta x\|<10^{-6}$ | 1000 | максимум итераций($C$)      |  $3.0 \cdot 10^{-11}$ по $f$ |
| 7. geom, старт $(10,5)$ | geom | $\|\Delta x\|<10^{-6}$ | 1000 | сошлось к $(3.82,\,0)$      |                            — |
| **8. главный, $\delta=10^{-4}$** | **geom** | $\|\Delta x\|<10^{-4}$ | **918** | **сошлось ($A$)**           | $\mathbf{7.0 \cdot 10^{-5}}$ |

### Комментарии

**Эксперимент 1.** Подтверждает необходимость нормировки или малого $a_0$: без них точки за несколько шагов улетают в область, где значения функции превышают размеры типов данных.

**Эксперимент 2.** Линейное убывание шага $a_k = a_0 / (1 + 0.01 k)$ слишком медленное, чтобы уменьшить осцилляции около минимума: к 1000-й итерации $a_k \approx 0.09$, что превышает $\delta = 10^{-6}$ на пять порядков. Метод не успевает сойтись.

**Эксперимент 3.** Постоянный шаг $a_k = 0.1$ при нормированном градиенте даёт длину шага ровно $0.1$ на каждой итерации (что и видно: $|\Delta x| = 0.1$ в выводе). Около минимума точка прыгает между симметричными окрестностями точек $A$ и $C$, не сходясь ни к одной из них.

**Эксперимент 4.** Геометрическая прогрессия $a_k = 0.99^k$ обеспечивает экспоненциальное уменьшение шага. С $\delta = 10^{-3}$ метод останавливается за 689 итераций. Абсолютная погрешность по функции составляет $1.8 \cdot 10^{-8}$.

**Эксперимент 5.** Сравнение двух критериев при одинаковых параметрах шага и $\varepsilon = \delta = 10^{-6}$:
- Критерий $\|\Delta x\|$ не успевает сработать за 1000 итераций (для срабатывания нужно $a_k < 10^{-6}$, что наступает на $\approx 1374$-й итерации), но точка всё равно оказывается рядом с $A$.
- Критерий $|\Delta f|$ срабатывает за 116 итераций далеко от минимума: функция в этой области очень пологая, $|\Delta f|$ становится мал задолго до приближения к точке экстремума. Это ложное срабатывание, демонстрирующее недостаток критерия по приращению функции на поверхностях с плоскими участками.

**Эксперимент 6.** При $(7, 7)$ метод сходится к точке $C = (-x_a, y_a)$, симметричной к $A$. Значение функции совпадает с точным до $10^{-11}$. Ошибка по точке относительно $A$ велика ($\approx 0.39$) из-за выбора $A$ в качестве исследуемой точки. Найдена эквивалентная точка минимума.

**Эксперимент 7.** При $(10, 5)$ метод сходится к точке $\approx (3.82, 0)$ на оси $X$. Метод нашёл представителя бесконечного множества нестрогих минимумов вне эллипса $\ln(8x^2 + 3y^2) > 0$.

## Главный эксперимент 8

Параметры:
- стартовая точка $(10, 10)$;
- $a_0 = 1$, $q = 0.99$, режим геометрического убывания шага;
- нормированный градиент;
- критерий остановки $\|\Delta x\| < \delta$, $\delta = 10^{-4}$;
- максимальное число итераций = 1000.

**Обоснование выбора $\delta$.** При нормированном градиенте и геометрическом убывании длина шага равна $\|\Delta x_k\| = a_k = a_0 q^k$. Условие остановки $\|\Delta x_k\| < \delta$ срабатывает, когда $a_0 q^k < \delta$, то есть на итерации
$$k > \frac{\ln \delta - \ln a_0}{\ln q}.$$
При $a_0 = 1$, $q = 0.99$, $\delta = 10^{-4}$: $k > \ln(10^{-4}) / \ln(0.99) \approx 916.4$, что согласуется с фактическим числом итераций (918). Меньшие $\delta$ (например, $10^{-5}$ требует $\approx 1146$ итераций) выходят за установленный лимит. Большие $\delta$ ($10^{-3}$ - 689 итераций) дают сравнимую, но меньшую точность по значению функции и точке. Выбор критерия по $\Delta f$ неоптимален для данной задачи из-за плоских участков в окрестности минимума, что подтверждается экспериментом 5.

**Результат.**
- Найденная точка: $(0.194768,\ 0.317964)$.
- Точная точка: $(0.194700,\ 0.317944)$.
- Отклонение по точке: $7.0 \cdot 10^{-5}$.
- Найденное значение: $-1.916038 \cdot 10^{-3}$.
- Точное значение: $-1.916039 \cdot 10^{-3}$.
- Отклонение по значению: $8.0 \cdot 10^{-10}$.
- Время работы: $\approx 8$ мс.

### Визуализация

Графики построены в Desmos 3D на основе экспортированной траектории

![Поверхность функции и траектория спуска](surface_big_scale.png)
![Поверхность функции и траектория в окрестности минимума](surface_low_scale.png)
*Красные точки - траектория спуска, зелёная - точная стационарная точка $A$.*

![Линии уровня и траектория спуска](level_lines_big_scale.png)
![Линии уровня и траектория спуска в окрестности минимума](level_lines_low_scale.png)
*Зеленые точки - траектория спуска, красная - точная стационарная точка $A$.*

*На всех картинках видно, что в окрестности А траектория зигзагообразно приближается к минимуму, поскольку длина каждого шага больше, чем фактическое расстояние до минимума.*

## Вывод

Метод градиентного спуска с нормированным градиентом и геометрическим убыванием шага позволил локализовать строгий локальный минимум функции $f(x, y) = x^2 y^2 \ln(8x^2 + 3y^2)$ с отклонением от точного решения порядка $10^{-4}$ по координатам и $10^{-9}$ по значению функции за 918 итераций (около 8 мс). Полученный результат согласуется с аналитическим: найденная точка лежит в окрестности $A = \bigl(\tfrac{1}{4 e^{1/4}}, \tfrac{1}{\sqrt 6\, e^{1/4}}\bigr) \approx (0.1947,\ 0.3179)$, найденное значение совпадает с $f(A) = -\tfrac{1}{192e} \approx -1.916 \cdot 10^{-3}$ до десятого знака.

Серия дополнительных экспериментов показала: критерий $|\Delta f|$ ненадёжен на функциях с плоскими участками (даёт ложные срабатывания вдали от минимума), критерий $\|\Delta x\|$ при геометрическом шаге предсказуемо срабатывает по формуле $k > \ln \delta / \ln q$, и метод корректно сходится к разным стационарным точкам в зависимости от стартовой точки.

- Запуск в Google Colab: [ссылка](https://drive.google.com/file/d/1LGy3Oshks2qyV9lprXnOiLByUubhrbe8/view?usp=sharing)
- Интерактивный 2D-график линий уровня: [ссылка](https://www.desmos.com/calculator/rtwqc572nr)
- Интерактивный 3D-график поверхности: [ссылка](https://www.desmos.com/3d/lle7gw77xt)

In [12]:
export_trajectory("traj.tsv")